In [1]:
# --- Setup & connection ---
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import numpy as np

env_path = Path(r"C:\Dev\india-air-quality-intel\.env")
load_dotenv(dotenv_path=env_path, override=True)
user = os.getenv("MYSQL_USER"); password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST"); port = os.getenv("MYSQL_PORT"); database = os.getenv("MYSQL_DATABASE")
engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}")

check = pd.read_sql("SELECT COUNT(*) AS n FROM fact_kaggle_historical", engine)
assert check['n'][0] == 1140696
print("Connection OK:", check['n'][0])

# --- Stable panel ---
query = """
SELECT ds.station_id, ds.source_station_key, ds.station_name, dc.city_name,
       COUNT(DISTINCT YEAR(fka.reading_date)) AS years_present
FROM fact_kaggle_daily_aqi fka
JOIN dim_station ds ON fka.station_id = ds.station_id
JOIN dim_city dc   ON ds.city_id = dc.city_id
WHERE ds.source_system = 'kaggle'
GROUP BY ds.station_id, ds.source_station_key, ds.station_name, dc.city_name
HAVING years_present = 6
ORDER BY dc.city_name, ds.station_name;
"""
stable_panel = pd.read_sql(query, engine)
modeling_panel = stable_panel[stable_panel['source_station_key'] != 'GJ001']
station_ids = tuple(modeling_panel['station_id'].tolist())
print("Stable panel:", len(stable_panel), "| Modeling panel (no GJ001):", len(modeling_panel))

# --- Raw station-day rows ---
raw_query = f"""
SELECT ds.station_id, dc.city_name, fka.reading_date, fka.aqi_value, fka.flag_missing
FROM fact_kaggle_daily_aqi fka
JOIN dim_station ds ON fka.station_id = ds.station_id
JOIN dim_city dc   ON ds.city_id = dc.city_id
WHERE fka.station_id IN {station_ids}
ORDER BY dc.city_name, ds.station_id, fka.reading_date;
"""
raw = pd.read_sql(raw_query, engine)
raw['reading_date'] = pd.to_datetime(raw['reading_date'])
print("Raw rows:", len(raw), "| Stations:", raw['station_id'].nunique())

# --- City-day aggregation ---
city_day = raw.groupby(['city_name', 'reading_date']).agg(
    aqi_mean=('aqi_value', 'mean'),
    stations_total=('station_id', 'count'),
    stations_valid=('aqi_value', lambda x: x.notna().sum())
).reset_index()
city_day['pct_stations_valid'] = (city_day['stations_valid'] / city_day['stations_total'] * 100).round(1)
city_day['aqi_missing'] = city_day['aqi_mean'].isna()
print("City-day rows:", len(city_day))

# --- Anomaly integration (consolidated — this is the piece that broke) ---
anomaly_query = f"""
SELECT fka.station_id, fka.reading_date, fkaf.is_anomaly, fkaf.modified_z, fkaf.baseline_note
FROM fact_kaggle_daily_aqi fka
JOIN fact_kaggle_anomaly_flags fkaf ON fka.daily_aqi_id = fkaf.daily_aqi_id
WHERE fka.station_id IN {station_ids}
"""
anomalies = pd.read_sql(anomaly_query, engine)
anomalies['reading_date'] = pd.to_datetime(anomalies['reading_date'])
raw_scored = raw.merge(anomalies, on=['station_id', 'reading_date'], how='left')

anomaly_rollup = raw_scored.groupby(['city_name', 'reading_date']).agg(
    stations_scored=('is_anomaly', lambda x: x.notna().sum()),
    stations_anomalous=('is_anomaly', lambda x: (x == 1).sum())
).reset_index()
anomaly_rollup['pct_anomalous'] = (anomaly_rollup['stations_anomalous'] / anomaly_rollup['stations_scored'] * 100).round(1)
anomaly_rollup['exclude_from_training'] = (
    anomaly_rollup['stations_anomalous'] > (anomaly_rollup['stations_scored'] / 2)
)
print("Anomaly rollup columns:", anomaly_rollup.columns.tolist())

# --- Final merge ---
final_dataset = city_day.merge(
    anomaly_rollup[['city_name', 'reading_date', 'stations_scored', 'stations_anomalous',
                     'pct_anomalous', 'exclude_from_training']],
    on=['city_name', 'reading_date'], how='left'
)

known_bad_windows = [
    ('Delhi',   '2017-06-01', '2017-09-26'),
    ('Patna',   '2017-06-01', '2017-09-26'),
    ('Lucknow', '2018-02-15', '2018-06-12'),
    ('Mumbai',  '2015-01-01', '2017-12-31'),
]
final_dataset['in_known_bad_window'] = False
for city, start, end in known_bad_windows:
    mask = (final_dataset['city_name'] == city) & (final_dataset['reading_date'] >= start) & (final_dataset['reading_date'] <= end)
    final_dataset.loc[mask, 'in_known_bad_window'] = True

print("Final dataset shape:", final_dataset.shape)
print(final_dataset.isna().sum())

final_dataset.to_csv('../data/processed/phase10_city_day_forecasting_base.csv', index=False)
print("Saved successfully.")

Connection OK: 1140696
Stable panel: 24 | Modeling panel (no GJ001): 23
Raw rows: 44804 | Stations: 23
City-day rows: 13909
Anomaly rollup columns: ['city_name', 'reading_date', 'stations_scored', 'stations_anomalous', 'pct_anomalous', 'exclude_from_training']
Final dataset shape: (13909, 12)
city_name                   0
reading_date                0
aqi_mean                 2896
stations_total              0
stations_valid              0
pct_stations_valid          0
aqi_missing                 0
stations_scored             0
stations_anomalous          0
pct_anomalous            2896
exclude_from_training       0
in_known_bad_window         0
dtype: int64
Saved successfully.


In [2]:
final_dataset['reading_date'] = pd.to_datetime(final_dataset['reading_date'])
TRAIN_END = '2019-12-31'
final_dataset['split'] = final_dataset['reading_date'].apply(lambda d: 'train' if d <= pd.Timestamp(TRAIN_END) else 'test')

final_dataset = final_dataset.sort_values(['city_name', 'reading_date']).reset_index(drop=True)
final_dataset['persistence_pred'] = final_dataset.groupby('city_name')['aqi_mean'].shift(1)

final_dataset['month'] = final_dataset['reading_date'].dt.month
train_mask = final_dataset['split'] == 'train'
seasonal_baseline = (
    final_dataset[train_mask].groupby(['city_name', 'month'])['aqi_mean'].median()
    .reset_index().rename(columns={'aqi_mean': 'seasonal_median_pred'})
)
final_dataset = final_dataset.merge(seasonal_baseline, on=['city_name', 'month'], how='left')
print(final_dataset.groupby(['city_name', 'split']).size().unstack())

split      test  train
city_name             
Bengaluru   183   1826
Chennai     183   1826
Delhi       183   1826
Hyderabad   183   1823
Lucknow     183   1826
Mumbai      183   1826
Patna       183   1675


In [3]:
final_dataset['day_of_year'] = final_dataset['reading_date'].dt.dayofyear
for k in [1, 2, 3]:
    final_dataset[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * final_dataset['day_of_year'] / 365.25)
    final_dataset[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * final_dataset['day_of_year'] / 365.25)

final_dataset = final_dataset.sort_values(['city_name', 'reading_date']).reset_index(drop=True)
for lag in [1, 7]:
    final_dataset[f'lag_{lag}'] = final_dataset.groupby('city_name')['aqi_mean'].shift(lag)

final_dataset['roll_mean_7'] = (
    final_dataset.groupby('city_name')['aqi_mean'].shift(1).rolling(7, min_periods=3).mean().reset_index(level=0, drop=True)
)
final_dataset['roll_mean_30'] = (
    final_dataset.groupby('city_name')['aqi_mean'].shift(1).rolling(30, min_periods=10).mean().reset_index(level=0, drop=True)
)

feature_cols = ['fourier_sin_1','fourier_cos_1','fourier_sin_2','fourier_cos_2','fourier_sin_3','fourier_cos_3',
                'lag_1','lag_7','roll_mean_7','roll_mean_30']
target_col = 'aqi_mean'

model_df = final_dataset.dropna(subset=feature_cols + [target_col]).copy()
train_final = model_df[(model_df['split']=='train') & (~model_df['in_known_bad_window']) & (~model_df['exclude_from_training'].fillna(False))]
test_final = model_df[model_df['split']=='test']
print("Train rows:", len(train_final), "| Test rows:", len(test_final))
print(train_final.groupby('city_name').size())
print(test_final.groupby('city_name').size())

Train rows: 9305 | Test rows: 1074
city_name
Bengaluru    1654
Chennai      1602
Delhi        1645
Hyderabad    1222
Lucknow      1519
Mumbai        519
Patna        1144
dtype: int64
city_name
Bengaluru    183
Chennai      183
Delhi        183
Hyderabad    169
Lucknow      183
Mumbai        95
Patna         78
dtype: int64


In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

predictions = []

for city in train_final['city_name'].unique():
    train_city = train_final[train_final['city_name'] == city]
    test_city = test_final[test_final['city_name'] == city]
    if len(test_city) == 0:
        continue

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_city[feature_cols])
    X_test = scaler.transform(test_city[feature_cols])

    model = Ridge(alpha=1.0)
    model.fit(X_train, train_city[target_col])

    result = test_city[['city_name', 'reading_date', 'aqi_mean', 'persistence_pred', 'seasonal_median_pred']].copy()
    result['ridge_pred'] = model.predict(X_test)
    predictions.append(result)

    coef_summary = dict(zip(feature_cols, model.coef_.round(1)))
    print(city, "| lag_1:", coef_summary['lag_1'], "| lag_7:", coef_summary['lag_7'], "| roll_mean_7:", coef_summary['roll_mean_7'])

ridge_results = pd.concat(predictions, ignore_index=True)
print("Total predictions:", len(ridge_results))

Bengaluru | lag_1: 15.0 | lag_7: -0.9 | roll_mean_7: 7.6
Chennai | lag_1: 24.4 | lag_7: -1.6 | roll_mean_7: 2.4
Delhi | lag_1: 75.6 | lag_7: 3.9 | roll_mean_7: -0.1
Hyderabad | lag_1: 27.1 | lag_7: 1.5 | roll_mean_7: -0.2
Lucknow | lag_1: 82.4 | lag_7: 10.4 | roll_mean_7: -2.9
Mumbai | lag_1: 21.8 | lag_7: 4.1 | roll_mean_7: -4.1
Patna | lag_1: 74.5 | lag_7: 2.1 | roll_mean_7: -0.8
Total predictions: 1074


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

eval_results = []
for city in ridge_results['city_name'].unique():
    city_data = ridge_results[ridge_results['city_name'] == city].dropna(
        subset=['aqi_mean', 'persistence_pred', 'seasonal_median_pred', 'ridge_pred']
    )
    for col, label in [('persistence_pred','persistence'), ('seasonal_median_pred','seasonal_median'), ('ridge_pred','ridge')]:
        rmse = np.sqrt(mean_squared_error(city_data['aqi_mean'], city_data[col]))
        mae = mean_absolute_error(city_data['aqi_mean'], city_data[col])
        r2 = r2_score(city_data['aqi_mean'], city_data[col])
        eval_results.append({'city_name': city, 'model': label, 'n': len(city_data), 'rmse': round(rmse,1), 'mae': round(mae,1), 'r2': round(r2,3)})

eval_df = pd.DataFrame(eval_results)
print(eval_df.pivot(index='city_name', columns='model', values=['rmse','mae','r2']))

                 rmse                               mae                        \
model     persistence ridge seasonal_median persistence ridge seasonal_median   
city_name                                                                       
Bengaluru        16.6  14.6            21.4        12.0  11.6            16.9   
Chennai          19.1  19.1            46.5        12.2  14.4            41.8   
Delhi            41.4  43.9           126.1        30.9  36.8           111.3   
Hyderabad        14.7  18.7            39.4        10.1  15.3            34.2   
Lucknow          50.8  47.5            88.9        36.9  36.9            73.9   
Mumbai           19.0  18.4            37.1        10.9  12.1            32.1   
Patna            52.6  48.3            78.9        39.9  38.2            61.8   

                   r2                         
model     persistence  ridge seasonal_median  
city_name                                     
Bengaluru       0.282  0.443          -0.185  
C

In [6]:
print(eval_df.pivot(index='city_name', columns='model', values='n'))

model      persistence  ridge  seasonal_median
city_name                                     
Bengaluru          183    183              183
Chennai            183    183              183
Delhi              183    183              183
Hyderabad          169    169              169
Lucknow            183    183              183
Mumbai              95     95               95
Patna               78     78               78


In [7]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 10.0, 50.0, 100.0]
best_alphas = {}

for city in train_final['city_name'].unique():
    train_city = train_final[train_final['city_name'] == city].sort_values('reading_date')
    val_start = int(len(train_city) * 0.8)
    inner_train, inner_val = train_city.iloc[:val_start], train_city.iloc[val_start:]

    best_alpha, best_rmse = None, np.inf
    for a in alphas:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(inner_train[feature_cols])
        X_val = scaler.transform(inner_val[feature_cols])
        m = Ridge(alpha=a).fit(X_tr, inner_train[target_col])
        rmse = np.sqrt(mean_squared_error(inner_val[target_col], m.predict(X_val)))
        if rmse < best_rmse:
            best_rmse, best_alpha = rmse, a

    best_alphas[city] = best_alpha
    print(city, "| best alpha:", best_alpha, "| inner val RMSE:", round(best_rmse, 1))

Bengaluru | best alpha: 50.0 | inner val RMSE: 21.6
Chennai | best alpha: 0.001 | inner val RMSE: 29.2
Delhi | best alpha: 0.001 | inner val RMSE: 49.4
Hyderabad | best alpha: 0.001 | inner val RMSE: 22.4
Lucknow | best alpha: 0.001 | inner val RMSE: 38.3
Mumbai | best alpha: 100.0 | inner val RMSE: 10.6
Patna | best alpha: 1.0 | inner val RMSE: 35.6


In [8]:
predictions_tuned = []
for city in train_final['city_name'].unique():
    train_city = train_final[train_final['city_name'] == city]
    test_city = test_final[test_final['city_name'] == city]
    if len(test_city) == 0:
        continue

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_city[feature_cols])
    X_test = scaler.transform(test_city[feature_cols])

    model = Ridge(alpha=best_alphas[city])
    model.fit(X_train, train_city[target_col])

    result = test_city[['city_name', 'reading_date', 'aqi_mean', 'persistence_pred', 'seasonal_median_pred']].copy()
    result['ridge_pred'] = model.predict(X_test)
    predictions_tuned.append(result)

ridge_results_tuned = pd.concat(predictions_tuned, ignore_index=True)

eval_results_tuned = []
for city in ridge_results_tuned['city_name'].unique():
    city_data = ridge_results_tuned[ridge_results_tuned['city_name'] == city].dropna(
        subset=['aqi_mean', 'persistence_pred', 'ridge_pred']
    )
    for col, label in [('persistence_pred','persistence'), ('ridge_pred','ridge_tuned')]:
        rmse = np.sqrt(mean_squared_error(city_data['aqi_mean'], city_data[col]))
        eval_results_tuned.append({'city_name': city, 'model': label, 'rmse': round(rmse,1), 'alpha': best_alphas[city]})

print(pd.DataFrame(eval_results_tuned).pivot(index='city_name', columns='model', values='rmse'))

model      persistence  ridge_tuned
city_name                          
Bengaluru         16.6         14.6
Chennai           19.1         19.1
Delhi             41.4         43.8
Hyderabad         14.7         18.7
Lucknow           50.8         47.6
Mumbai            19.0         19.8
Patna             52.6         48.3


In [9]:
# re-attach the anomaly flag to the tuned ridge results
anomaly_lookup = final_dataset[['city_name', 'reading_date', 'exclude_from_training', 'stations_anomalous']].copy()
ridge_eval = ridge_results_tuned.merge(anomaly_lookup, on=['city_name', 'reading_date'], how='left')
ridge_eval['is_anomaly_day'] = ridge_eval['stations_anomalous'] > 0  # any station anomalous = a real irregular day worth isolating

reliability = []
for city in ridge_eval['city_name'].unique():
    for is_anom, label in [(False, 'normal'), (True, 'anomaly')]:
        subset = ridge_eval[(ridge_eval['city_name'] == city) & (ridge_eval['is_anomaly_day'] == is_anom)].dropna(
            subset=['aqi_mean', 'persistence_pred', 'ridge_pred']
        )
        if len(subset) == 0:
            continue
        p_rmse = np.sqrt(mean_squared_error(subset['aqi_mean'], subset['persistence_pred']))
        r_rmse = np.sqrt(mean_squared_error(subset['aqi_mean'], subset['ridge_pred']))
        reliability.append({'city_name': city, 'day_type': label, 'n': len(subset), 'persistence_rmse': round(p_rmse,1), 'ridge_rmse': round(r_rmse,1)})

reliability_df = pd.DataFrame(reliability)
print(reliability_df.pivot(index='city_name', columns='day_type', values=['n','persistence_rmse','ridge_rmse']))

                n        persistence_rmse        ridge_rmse       
day_type  anomaly normal          anomaly normal    anomaly normal
city_name                                                         
Bengaluru     7.0  176.0             32.3   15.7       31.4   13.6
Chennai       3.0  180.0             60.0   17.6       62.9   17.5
Delhi         NaN  183.0              NaN   41.4        NaN   43.8
Hyderabad     3.0  166.0             30.2   14.3       49.8   17.7
Lucknow       3.0  180.0            118.5   48.8      112.4   45.7
Mumbai        NaN   95.0              NaN   19.0        NaN   19.8
Patna         NaN   78.0              NaN   52.6        NaN   48.3


In [10]:
def aqi_category(val):
    if pd.isna(val): return np.nan
    if val <= 50: return 'Good'
    elif val <= 100: return 'Satisfactory'
    elif val <= 200: return 'Moderate'
    elif val <= 300: return 'Poor'
    elif val <= 400: return 'Very Poor'
    else: return 'Severe'

ridge_eval['actual_category'] = ridge_eval['aqi_mean'].apply(aqi_category)
ridge_eval['ridge_category'] = ridge_eval['ridge_pred'].apply(aqi_category)
ridge_eval['persistence_category'] = ridge_eval['persistence_pred'].apply(aqi_category)

ridge_eval['ridge_category_match'] = ridge_eval['actual_category'] == ridge_eval['ridge_category']
ridge_eval['persistence_category_match'] = ridge_eval['actual_category'] == ridge_eval['persistence_category']

cat_accuracy = ridge_eval.dropna(subset=['actual_category']).groupby('city_name')[
    ['ridge_category_match', 'persistence_category_match']
].mean().round(3)
print(cat_accuracy)

# how far off, when wrong? (band-distance, not just right/wrong)
bands = ['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']
ridge_eval['actual_band_idx'] = ridge_eval['actual_category'].apply(lambda c: bands.index(c) if pd.notna(c) else np.nan)
ridge_eval['ridge_band_idx'] = ridge_eval['ridge_category'].apply(lambda c: bands.index(c) if pd.notna(c) else np.nan)
ridge_eval['band_distance'] = (ridge_eval['actual_band_idx'] - ridge_eval['ridge_band_idx']).abs()

print(ridge_eval.groupby('city_name')['band_distance'].value_counts().unstack(fill_value=0))

           ridge_category_match  persistence_category_match
city_name                                                  
Bengaluru                 0.721                       0.705
Chennai                   0.732                       0.781
Delhi                     0.634                       0.699
Hyderabad                 0.740                       0.858
Lucknow                   0.574                       0.645
Mumbai                    0.842                       0.800
Patna                     0.679                       0.654
band_distance    0   1  2
city_name                
Bengaluru      132  51  0
Chennai        134  49  0
Delhi          116  67  0
Hyderabad      125  44  0
Lucknow        105  74  4
Mumbai          80  15  0
Patna           53  23  2


In [11]:
big_misses = ridge_eval[ridge_eval['band_distance'] >= 2][
    ['city_name', 'reading_date', 'aqi_mean', 'ridge_pred', 'actual_category', 'ridge_category', 'is_anomaly_day']
]
print(big_misses)

     city_name reading_date  aqi_mean  ridge_pred actual_category  \
726    Lucknow   2020-01-09     198.5  352.238449        Moderate   
734    Lucknow   2020-01-17     143.5  304.521019        Moderate   
799    Lucknow   2020-03-22      83.0  208.005542    Satisfactory   
803    Lucknow   2020-03-26      97.0  212.696528    Satisfactory   
1037     Patna   2020-02-16      87.0  213.531778    Satisfactory   
1044     Patna   2020-02-28     302.0  198.794146       Very Poor   

     ridge_category  is_anomaly_day  
726       Very Poor           False  
734       Very Poor            True  
799            Poor           False  
803            Poor           False  
1037           Poor           False  
1044       Moderate           False  


In [12]:
final_dataset = final_dataset.sort_values(['city_name', 'reading_date']).reset_index(drop=True)
final_dataset['target_3day'] = final_dataset.groupby('city_name')['aqi_mean'].shift(-3)

# persistence-3day baseline: naive forecast is still "today's value", now compared 3 days ahead
final_dataset['persistence_3day_pred'] = final_dataset['aqi_mean']

model_df_3day = final_dataset.dropna(subset=feature_cols + ['target_3day']).copy()
train_3day = model_df_3day[(model_df_3day['split']=='train') & (~model_df_3day['in_known_bad_window']) & (~model_df_3day['exclude_from_training'].fillna(False))]
test_3day = model_df_3day[model_df_3day['split']=='test']
print("3-day train rows:", len(train_3day), "| test rows:", len(test_3day))
print(train_3day.groupby('city_name').size())
print(test_3day.groupby('city_name').size())

3-day train rows: 9242 | test rows: 1041
city_name
Bengaluru    1646
Chennai      1591
Delhi        1645
Hyderabad    1218
Lucknow      1509
Mumbai        504
Patna        1129
dtype: int64
city_name
Bengaluru    180
Chennai      180
Delhi        180
Hyderabad    163
Lucknow      180
Mumbai        86
Patna         72
dtype: int64


In [13]:
best_alphas_3day = {}
for city in train_3day['city_name'].unique():
    train_city = train_3day[train_3day['city_name'] == city].sort_values('reading_date')
    val_start = int(len(train_city) * 0.8)
    inner_train, inner_val = train_city.iloc[:val_start], train_city.iloc[val_start:]

    best_alpha, best_rmse = None, np.inf
    for a in [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0]:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(inner_train[feature_cols])
        X_val = scaler.transform(inner_val[feature_cols])
        m = Ridge(alpha=a).fit(X_tr, inner_train['target_3day'])
        rmse = np.sqrt(mean_squared_error(inner_val['target_3day'], m.predict(X_val)))
        if rmse < best_rmse:
            best_rmse, best_alpha = rmse, a
    best_alphas_3day[city] = best_alpha
    print(city, "| best alpha:", best_alpha, "| inner val RMSE:", round(best_rmse, 1))

Bengaluru | best alpha: 100.0 | inner val RMSE: 25.5
Chennai | best alpha: 0.001 | inner val RMSE: 43.6
Delhi | best alpha: 100.0 | inner val RMSE: 79.5
Hyderabad | best alpha: 1.0 | inner val RMSE: 35.3
Lucknow | best alpha: 100.0 | inner val RMSE: 58.9
Mumbai | best alpha: 100.0 | inner val RMSE: 13.4
Patna | best alpha: 100.0 | inner val RMSE: 47.9


In [14]:
# Seasonal baseline keyed to the TARGET date's month, not today's
final_dataset['target_date_3day'] = final_dataset['reading_date'] + pd.Timedelta(days=3)
final_dataset['target_month_3day'] = final_dataset['target_date_3day'].dt.month

seasonal_baseline_3day = seasonal_baseline.rename(
    columns={'month': 'target_month_3day', 'seasonal_median_pred': 'seasonal_median_pred_3day'}
)
final_dataset = final_dataset.merge(seasonal_baseline_3day, on=['city_name', 'target_month_3day'], how='left')

# Re-slice train/test now that new columns exist (row counts unchanged from before)
model_df_3day = final_dataset.dropna(subset=feature_cols + ['target_3day']).copy()
train_3day = model_df_3day[(model_df_3day['split']=='train') & (~model_df_3day['in_known_bad_window']) & (~model_df_3day['exclude_from_training'].fillna(False))]
test_3day = model_df_3day[model_df_3day['split']=='test']

predictions_3day = []
for city in train_3day['city_name'].unique():
    train_city = train_3day[train_3day['city_name'] == city]
    test_city = test_3day[test_3day['city_name'] == city]
    if len(test_city) == 0:
        continue
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_city[feature_cols])
    X_test = scaler.transform(test_city[feature_cols])
    model = Ridge(alpha=best_alphas_3day[city])
    model.fit(X_train, train_city['target_3day'])

    result = test_city[['city_name', 'reading_date', 'target_3day', 'aqi_mean', 'seasonal_median_pred_3day']].copy()
    result['persistence_3day_pred'] = result['aqi_mean']  # naive: today's value stands in for 3 days ahead
    result['ridge_3day_pred'] = model.predict(X_test)
    predictions_3day.append(result)

ridge_3day_results = pd.concat(predictions_3day, ignore_index=True)

eval_3day = []
for city in ridge_3day_results['city_name'].unique():
    d = ridge_3day_results[ridge_3day_results['city_name']==city].dropna(
        subset=['target_3day','persistence_3day_pred','ridge_3day_pred','seasonal_median_pred_3day']
    )
    for col,label in [('persistence_3day_pred','persistence'), ('ridge_3day_pred','ridge'), ('seasonal_median_pred_3day','seasonal')]:
        rmse = np.sqrt(mean_squared_error(d['target_3day'], d[col]))
        eval_3day.append({'city_name':city,'model':label,'n':len(d),'rmse':round(rmse,1)})

print(pd.DataFrame(eval_3day).pivot(index='city_name', columns='model', values='rmse'))

model      persistence  ridge  seasonal
city_name                              
Bengaluru         19.0   17.0      21.5
Chennai           30.4   27.3      46.1
Delhi             70.4   72.1     126.4
Hyderabad         29.0   29.9      38.8
Lucknow           72.4   62.2      89.3
Mumbai            24.8   25.6      35.5
Patna             80.0   64.3      84.8


In [15]:
best_model_1day = {
    'Bengaluru': 'ridge', 'Chennai': 'ridge', 'Delhi': 'persistence',
    'Hyderabad': 'persistence', 'Lucknow': 'ridge', 'Mumbai': 'persistence', 'Patna': 'ridge'
}
best_model_3day = {
    'Bengaluru': 'ridge', 'Chennai': 'ridge', 'Delhi': 'persistence',
    'Hyderabad': 'persistence', 'Lucknow': 'ridge', 'Mumbai': 'persistence', 'Patna': 'ridge'
}
print("Same routing at both horizons:", best_model_1day == best_model_3day)

Same routing at both horizons: True


In [16]:
router = best_model_1day  # identical at both horizons, confirmed last step

# --- 1-day routed forecast ---
ridge_eval['routed_pred_1day'] = ridge_eval.apply(
    lambda r: r['ridge_pred'] if router[r['city_name']] == 'ridge' else r['persistence_pred'], axis=1
)

# --- 3-day routed forecast ---
ridge_3day_results['routed_pred_3day'] = ridge_3day_results.apply(
    lambda r: r['ridge_3day_pred'] if router[r['city_name']] == 'ridge' else r['persistence_3day_pred'], axis=1
)

# Verify: routed RMSE per city should exactly equal whichever model already won at that horizon
for label, df, target_col, pred_col in [
    ('1-day', ridge_eval, 'aqi_mean', 'routed_pred_1day'),
    ('3-day', ridge_3day_results, 'target_3day', 'routed_pred_3day'),
]:
    print(f"--- {label} ---")
    for city in df['city_name'].unique():
        d = df[df['city_name'] == city].dropna(subset=[target_col, pred_col])
        rmse = np.sqrt(mean_squared_error(d[target_col], d[pred_col]))
        print(city, "| routed RMSE:", round(rmse, 1), "| using:", router[city])

--- 1-day ---
Bengaluru | routed RMSE: 14.6 | using: ridge
Chennai | routed RMSE: 19.1 | using: ridge
Delhi | routed RMSE: 41.4 | using: persistence
Hyderabad | routed RMSE: 14.7 | using: persistence
Lucknow | routed RMSE: 47.6 | using: ridge
Mumbai | routed RMSE: 19.0 | using: persistence
Patna | routed RMSE: 48.3 | using: ridge
--- 3-day ---
Bengaluru | routed RMSE: 17.0 | using: ridge
Chennai | routed RMSE: 27.3 | using: ridge
Delhi | routed RMSE: 70.4 | using: persistence
Hyderabad | routed RMSE: 29.0 | using: persistence
Lucknow | routed RMSE: 62.2 | using: ridge
Mumbai | routed RMSE: 24.8 | using: persistence
Patna | routed RMSE: 64.1 | using: ridge


In [17]:
d_old = ridge_3day_results.dropna(subset=['target_3day','persistence_3day_pred','ridge_3day_pred','seasonal_median_pred_3day'])
d_new = ridge_3day_results.dropna(subset=['target_3day','routed_pred_3day'])
print("Old n (Patna):", len(d_old[d_old['city_name']=='Patna']))
print("New n (Patna):", len(d_new[d_new['city_name']=='Patna']))

Old n (Patna): 70
New n (Patna): 72


In [18]:
ridge_eval['routed_category_1day'] = ridge_eval['routed_pred_1day'].apply(aqi_category)
ridge_eval['actual_category_1day'] = ridge_eval['aqi_mean'].apply(aqi_category)

ridge_3day_results['routed_category_3day'] = ridge_3day_results['routed_pred_3day'].apply(aqi_category)
ridge_3day_results['actual_category_3day'] = ridge_3day_results['target_3day'].apply(aqi_category)

def priority_tier(category):
    if category in ['Good', 'Satisfactory']:
        return 'Low'
    elif category in ['Moderate', 'Poor']:
        return 'Watch'
    else:  # Very Poor, Severe
        return 'Alert'

ridge_eval['priority_1day'] = ridge_eval['routed_category_1day'].apply(priority_tier)
ridge_3day_results['priority_3day'] = ridge_3day_results['routed_category_3day'].apply(priority_tier)

print(ridge_eval['priority_1day'].value_counts())
print(ridge_3day_results['priority_3day'].value_counts())

priority_1day
Low      554
Watch    445
Alert     75
Name: count, dtype: int64
priority_3day
Watch    491
Low      475
Alert     75
Name: count, dtype: int64


In [19]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

def alert_metrics(df, actual_col, pred_col, label):
    d = df.dropna(subset=[actual_col, pred_col]).copy()
    actual_alert = d[actual_col].apply(priority_tier) == 'Alert'
    pred_alert = d[pred_col].apply(priority_tier) == 'Alert'

    tp = ((actual_alert) & (pred_alert)).sum()
    fn = ((actual_alert) & (~pred_alert)).sum()
    fp = ((~actual_alert) & (pred_alert)).sum()
    tn = ((~actual_alert) & (~pred_alert)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    print(f"{label} | real Alert days: {tp+fn} | TP: {tp} FN: {fn} FP: {fp} | precision: {round(precision,3) if pd.notna(precision) else 'n/a'} | recall: {round(recall,3) if pd.notna(recall) else 'n/a'}")
    return {'label': label, 'real_alerts': tp+fn, 'tp': tp, 'fn': fn, 'fp': fp, 'precision': precision, 'recall': recall}

print("=== 1-day, by city ===")
results_alert_1day = []
for city in ridge_eval['city_name'].unique():
    d = ridge_eval[ridge_eval['city_name'] == city]
    results_alert_1day.append(alert_metrics(d, 'aqi_mean', 'routed_pred_1day', city))

print("\n=== 3-day, by city ===")
results_alert_3day = []
for city in ridge_3day_results['city_name'].unique():
    d = ridge_3day_results[ridge_3day_results['city_name'] == city]
    results_alert_3day.append(alert_metrics(d, 'target_3day', 'routed_pred_3day', city))

=== 1-day, by city ===
Bengaluru | real Alert days: 183 | TP: 183 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Chennai | real Alert days: 183 | TP: 183 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Delhi | real Alert days: 183 | TP: 183 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Hyderabad | real Alert days: 169 | TP: 169 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Lucknow | real Alert days: 183 | TP: 183 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Mumbai | real Alert days: 95 | TP: 95 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Patna | real Alert days: 78 | TP: 78 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0

=== 3-day, by city ===
Bengaluru | real Alert days: 180 | TP: 180 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Chennai | real Alert days: 180 | TP: 180 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Delhi | real Alert days: 180 | TP: 180 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Hyderabad | real Alert days: 163 | TP: 163 FN: 0 FP: 0 | precision: 1.0 | recall: 1.0
Lucknow | real Alert days: 180 |

In [20]:
def alert_metrics(df, actual_col, pred_col, label):
    d = df.dropna(subset=[actual_col, pred_col]).copy()
    actual_alert = d[actual_col].apply(aqi_category).apply(priority_tier) == 'Alert'
    pred_alert = d[pred_col].apply(aqi_category).apply(priority_tier) == 'Alert'

    tp = ((actual_alert) & (pred_alert)).sum()
    fn = ((actual_alert) & (~pred_alert)).sum()
    fp = ((~actual_alert) & (pred_alert)).sum()
    tn = ((~actual_alert) & (~pred_alert)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    print(f"{label} | real Alert days: {tp+fn} | TP: {tp} FN: {fn} FP: {fp} | precision: {round(precision,3) if pd.notna(precision) else 'n/a'} | recall: {round(recall,3) if pd.notna(recall) else 'n/a'}")
    return {'label': label, 'real_alerts': tp+fn, 'tp': tp, 'fn': fn, 'fp': fp, 'precision': precision, 'recall': recall}

print("=== 1-day, by city ===")
results_alert_1day = []
for city in ridge_eval['city_name'].unique():
    d = ridge_eval[ridge_eval['city_name'] == city]
    results_alert_1day.append(alert_metrics(d, 'aqi_mean', 'routed_pred_1day', city))

print("\n=== 3-day, by city ===")
results_alert_3day = []
for city in ridge_3day_results['city_name'].unique():
    d = ridge_3day_results[ridge_3day_results['city_name'] == city]
    results_alert_3day.append(alert_metrics(d, 'target_3day', 'routed_pred_3day', city))

=== 1-day, by city ===
Bengaluru | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Chennai | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Delhi | real Alert days: 25 | TP: 17 FN: 8 FP: 9 | precision: 0.654 | recall: 0.68
Hyderabad | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Lucknow | real Alert days: 13 | TP: 5 FN: 8 FP: 6 | precision: 0.455 | recall: 0.385
Mumbai | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Patna | real Alert days: 38 | TP: 32 FN: 6 FP: 6 | precision: 0.842 | recall: 0.842

=== 3-day, by city ===
Bengaluru | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Chennai | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Delhi | real Alert days: 22 | TP: 9 FN: 13 FP: 16 | precision: 0.36 | recall: 0.409
Hyderabad | real Alert days: 0 | TP: 0 FN: 0 FP: 0 | precision: n/a | recall: n/a
Lucknow | real Alert days: 11 | TP: 2 FN: 9 FP: 5 

In [22]:
import os

# --- 1-day predictions, standardized schema ---
export_1day = ridge_eval[[
    'city_name', 'reading_date', 'aqi_mean', 'routed_pred_1day',
    'actual_category_1day', 'routed_category_1day', 'priority_1day',
    'is_anomaly_day'
]].copy()

export_1day = export_1day.rename(columns={
    'city_name': 'city',
    'aqi_mean': 'actual_aqi',
    'routed_pred_1day': 'predicted_aqi',
    'actual_category_1day': 'actual_category',
    'routed_category_1day': 'predicted_category',
    'priority_1day': 'priority'
})
export_1day['horizon'] = 1
export_1day['target_date'] = export_1day['reading_date'] + pd.Timedelta(days=1)
export_1day['model_used'] = export_1day['city'].map(router)

# --- 3-day predictions, standardized schema ---
export_3day = ridge_3day_results[[
    'city_name', 'reading_date', 'target_3day', 'routed_pred_3day',
    'actual_category_3day', 'routed_category_3day', 'priority_3day'
]].copy()

export_3day = export_3day.rename(columns={
    'city_name': 'city',
    'target_3day': 'actual_aqi',
    'routed_pred_3day': 'predicted_aqi',
    'actual_category_3day': 'actual_category',
    'routed_category_3day': 'predicted_category',
    'priority_3day': 'priority'
})
export_3day['horizon'] = 3
export_3day['target_date'] = export_3day['reading_date'] + pd.Timedelta(days=3)
export_3day['model_used'] = export_3day['city'].map(router)
export_3day['is_anomaly_day'] = np.nan  # not computed for the 3-day frame in Phase 10 — see note below

# --- Combine into one schema ---
common_cols = ['city', 'reading_date', 'target_date', 'horizon', 'model_used',
               'actual_aqi', 'predicted_aqi', 'actual_category', 'predicted_category',
               'priority', 'is_anomaly_day']

phase10_export = pd.concat([export_1day[common_cols], export_3day[common_cols]], ignore_index=True)

os.makedirs("../data/processed/phase10_test_predictions", exist_ok=True)
phase10_export.to_csv("../data/processed/phase10_test_predictions/phase10_test_predictions.csv", index=False)

print(phase10_export.shape)
print(phase10_export.groupby(['city', 'horizon']).size().unstack())
print(phase10_export['is_anomaly_day'].value_counts(dropna=False))

(2115, 11)
horizon      1    3
city               
Bengaluru  183  180
Chennai    183  180
Delhi      183  180
Hyderabad  169  163
Lucknow    183  180
Mumbai      95   86
Patna       78   72
is_anomaly_day
False    1058
NaN      1041
True       16
Name: count, dtype: int64
